In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 0. Setup
import os
import random
import shutil
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image, ImageOps

In [ ]:
# Constants
BASE_PATH = '/content/drive/MyDrive/Wheat_Images'    # original dataset root (per-class folders)
SAVE_ROOT = '/content/drive/MyDrive/Wheat_Final'    # where balanced train/val/test folders will be created
OUTPUT_BASE = '/content/drive/MyDrive'               # where model outputs and splits are saved
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_BASE, 'Wheat_Output')
SPLITS_DIR = os.path.join(OUTPUT_BASE, 'wheat_splits')

TARGET_PER_CLASS = 500
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE
RANDOM_STATE = 42

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_ROOT, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

In [ ]:
# 1. Define class names (must match folder names under BASE_PATH)
CLASS_NAMES = sorted([d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))])
if len(CLASS_NAMES) != NUM_CLASSES:
    print("WARNING: number of classes found in BASE_PATH (=) ", len(CLASS_NAMES))
print("Classes:", CLASS_NAMES)
CLASS_TO_INDEX = {c: i for i, c in enumerate(CLASS_NAMES)}

Classes: ['Crown and Root Rot', 'Healthy Wheat', 'Leaf Rust', 'Septoria', 'Stripe Rust']


In [ ]:
# 2. Build balanced list of file paths (exactly TARGET_PER_CLASS per class)
balanced_paths = []
balanced_labels = []

for cls in CLASS_NAMES:
    cls_dir = os.path.join(BASE_PATH, cls)
    files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.jfif'))]
    if len(files) == 0:
        raise ValueError(f"No images found for class '{cls}' in {cls_dir}")

    # If at least target, sample without replacement; else sample with replacement
    if len(files) >= TARGET_PER_CLASS:
        chosen = random.sample(files, TARGET_PER_CLASS)
    else:
        # fill by repeating random choices (keeps variety)
        chosen = list(files)
        while len(chosen) < TARGET_PER_CLASS:
            chosen.append(random.choice(files))
        # shuffle to mix
        random.shuffle(chosen)

    assert len(chosen) == TARGET_PER_CLASS
    balanced_paths.extend(chosen)
    balanced_labels.extend([CLASS_TO_INDEX[cls]] * TARGET_PER_CLASS)

# Global shuffle
combined = list(zip(balanced_paths, balanced_labels))
random.seed(RANDOM_STATE)
random.shuffle(combined)
balanced_paths, balanced_labels = zip(*combined)
balanced_paths = list(balanced_paths)
balanced_labels = list(balanced_labels)

print(f"Balanced dataset created: {len(balanced_paths)} images ({TARGET_PER_CLASS} per class)")

# Quick sanity: counts per class
from collections import Counter
print('Per-class counts (balanced):', Counter(balanced_labels))

Balanced dataset created: 2500 images (500 per class)
Per-class counts (balanced): Counter({3: 500, 0: 500, 4: 500, 2: 500, 1: 500})


In [ ]:
# 3. Stratified split: train 80% / val 10% / test 10% (uses BALANCED lists)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    balanced_paths, balanced_labels, test_size=0.20, stratify=balanced_labels, random_state=RANDOM_STATE)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=RANDOM_STATE)

print('Split sizes ->', 'Train:', len(train_paths), 'Val:', len(val_paths), 'Test:', len(test_paths))

# Save splits (so different runs / models use same split)
np.save(os.path.join(SPLITS_DIR, 'train_paths.npy'), np.array(train_paths, dtype=object))
np.save(os.path.join(SPLITS_DIR, 'train_labels.npy'), np.array(train_labels, dtype=np.int32))
np.save(os.path.join(SPLITS_DIR, 'val_paths.npy'), np.array(val_paths, dtype=object))
np.save(os.path.join(SPLITS_DIR, 'val_labels.npy'), np.array(val_labels, dtype=np.int32))
np.save(os.path.join(SPLITS_DIR, 'test_paths.npy'), np.array(test_paths, dtype=object))
np.save(os.path.join(SPLITS_DIR, 'test_labels.npy'), np.array(test_labels, dtype=np.int32))
print('Saved splits to', SPLITS_DIR)

Split sizes -> Train: 2000 Val: 250 Test: 250
Saved splits to /content/drive/MyDrive/wheat_splits


In [ ]:
# # 4. Copy images into Drive folders (train/val/test) - optional but helpful for inspection and reproducibility

# def copy_split(paths, labels, split_name):
#     split_dir = os.path.join(SAVE_ROOT, split_name)
#     os.makedirs(split_dir, exist_ok=True)
#     # create per-class folders
#     for cls in CLASS_NAMES:
#         os.makedirs(os.path.join(split_dir, cls), exist_ok=True)

#     for p, lbl in zip(paths, labels):
#         cls_name = CLASS_NAMES[lbl]
#         dst = os.path.join(split_dir, cls_name, os.path.basename(p))
#         # avoid overwriting duplicates by adding an index if file exists
#         if os.path.exists(dst):
#             base, ext = os.path.splitext(os.path.basename(p))
#             dst = os.path.join(split_dir, cls_name, f"{base}_{random.randint(0,9999)}{ext}")
#         shutil.copy(p, dst)

#     print(f"Copied {len(paths)} files to {split_dir}")

# # Run copy (comment out if you prefer not to duplicate files)
# copy_split(train_paths, train_labels, 'train')
# copy_split(val_paths, val_labels, 'val')
# copy_split(test_paths, test_labels, 'test')

KeyboardInterrupt: 

In [ ]:
# 5. Dataset helper functions (read, augment, preprocess)

def _read_and_decode(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.cast(image, tf.float32)
    return image, label

# Augment: resize -> random crop -> color jitter
def augment(image, label):
    # Resize so smaller side >= 244 then random crop 224x224
    image = tf.image.resize(image, [244, 244])
    image = tf.image.random_crop(image, size=[IMAGE_SIZE[0], IMAGE_SIZE[1], 3])

    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.08)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.image.random_saturation(image, lower=0.9, upper=1.1)
    image = tf.image.random_hue(image, max_delta=0.02)

    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

# Preprocess for val/test: resize with pad to keep aspect and center
def preprocess_only(image, label):
    image = tf.image.resize_with_pad(image, IMAGE_SIZE[0], IMAGE_SIZE[1])
    return image, label

# utility to create dataset from paths/labels
def make_dataset(paths, labels, training=True):
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=RANDOM_STATE)
    ds = ds.map(_read_and_decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(preprocess_only, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

# Create datasets
train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

# Quick sanity
for images, labels in train_ds.take(1):
    print('Train batch:', images.shape, labels.shape)
for images, labels in val_ds.take(1):
    print('Val batch:', images.shape, labels.shape)

Train batch: (32, 224, 224, 3) (32,)
Val batch: (32, 224, 224, 3) (32,)


In [ ]:
# 6. Model building + training function

def build_base_and_preprocess(model_name):
    model_name = model_name.lower()
    if 'efficientnetv2b0' in model_name or model_name=='efficientnetv2b0':
        base = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet', input_shape=(224,224,3))
        preprocess_fn = tf.keras.applications.efficientnet_v2.preprocess_input
    elif 'efficientnetb0' in model_name or model_name=='efficientnetb0':
        base = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224,224,3))
        preprocess_fn = tf.keras.applications.efficientnet.preprocess_input
    elif 'mobilenetv2' in model_name or model_name=='mobilenetv2':
        base = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(224,224,3))
        preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input
    elif 'resnet50' in model_name or model_name=='resnet50':
        base = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=(224,224,3))
        preprocess_fn = tf.keras.applications.resnet.preprocess_input
    else:
        raise ValueError('Unknown model')
    return base, preprocess_fn


# def train_model(model_name, head_epochs=10, fine_tune_epochs=20, unfreeze_percent=0.2):
#     print('\n' + '='*10, 'Training', model_name, '='*10)
#     base, preprocess_fn = build_base_and_preprocess(model_name)
#     base.trainable = False

#     inputs = tf.keras.Input(shape=(224,224,3))
#     # Apply model-specific preprocess as a Lambda; when loading the model later, pass custom_objects if needed
#     x = layers.Lambda(preprocess_fn, name='preprocess')(inputs)
#     x = base(x, training=False)
#     x = layers.GlobalAveragePooling2D()(x)
#     x = layers.Dropout(0.3)(x)
#     x = layers.Dense(256, activation='relu')(x)
#     x = layers.Dropout(0.25)(x)
#     outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

#     model = tf.keras.Model(inputs, outputs)
#     model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#     early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)
#     reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.3, verbose=1)

#     # Phase 1: train head
#     print('Phase 1 — train head')
#     history_head = model.fit(train_ds, validation_data=val_ds, epochs=head_epochs, callbacks=[early_stop, reduce_lr])

#     # Phase 2: fine-tune last `unfreeze_percent` of base
#     total_layers = len(base.layers)
#     n_unfreeze = max(1, int(total_layers * unfreeze_percent))
#     print('Unfreezing last', n_unfreeze, 'layers of the base model')
#     for layer in base.layers[-n_unfreeze:]:
#         layer.trainable = True

#     model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
#     print('Phase 2 — fine-tuning')
#     history_ft = model.fit(train_ds, validation_data=val_ds, epochs=fine_tune_epochs, callbacks=[early_stop, reduce_lr])

#     # Save model (h5)
#     filename = f"{model_name.lower()}_balanced_{TARGET_PER_CLASS}.h5"
#     model_path = os.path.join(MODEL_OUTPUT_DIR, filename)
#     model.save(model_path)
#     print('Saved model to', model_path)

#     # Also save Keras saved-model folder (optional)
#     saved_model_dir = os.path.join(MODEL_OUTPUT_DIR, f"{model_name.lower()}_balanced_{TARGET_PER_CLASS}_savedmodel")
#     tf.saved_model.save(model, saved_model_dir)
#     print('Saved SavedModel to', saved_model_dir)

#     return model, history_head, history_ft

In [ ]:
def train_model(model_name, head_epochs=10, fine_tune_epochs=20, unfreeze_percent=0.2):
    print('\n' + '='*12, f'Training {model_name}', '='*12)

    # -------------------------------------------------
    # Build base model + preprocess function
    # -------------------------------------------------
    base, preprocess_fn = build_base_and_preprocess(model_name)
    base.trainable = False

    # -------------------------------------------------
    # Build full model
    # -------------------------------------------------
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.layers.Lambda(preprocess_fn, name="preprocess")(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # -------------------------------------------------
    # Callbacks
    # -------------------------------------------------
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=3,
        factor=0.3,
        verbose=1
    )

    callbacks = [early_stop, reduce_lr]

    # -------------------------------------------------
    # Phase 1 — Train classification head
    # -------------------------------------------------
    print("Phase 1 — training head (base frozen)")
    history_head = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=head_epochs,
        callbacks=callbacks
    )

    # -------------------------------------------------
    # Phase 2 — Fine-tuning
    # -------------------------------------------------
    total_layers = len(base.layers)
    n_unfreeze = max(1, int(total_layers * unfreeze_percent))

    print(f"Unfreezing last {n_unfreeze} layers out of {total_layers}")

    for layer in base.layers[-n_unfreeze:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Phase 2 — fine-tuning")
    history_ft = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=fine_tune_epochs,
        callbacks=callbacks
    )

    # -------------------------------------------------
    # ✅ SAVE MODEL — KERAS FORMAT ONLY
    # -------------------------------------------------
    model_filename = f"{model_name.lower()}_balanced_{TARGET_PER_CLASS}.keras"
    model_path = os.path.join(MODEL_OUTPUT_DIR, model_filename)
    model.save(model_path)

    print("✅ Model saved to:", model_path)

    return model, history_head, history_ft


In [ ]:
# 7. Choose a model and train (single model run)
# Change model_name to switch to MobileNetV2 / ResNet50 / EfficientNetV2B0
model_name = 'MobileNetV2'
model, h1, h2 = train_model(model_name, head_epochs=10, fine_tune_epochs=20, unfreeze_percent=0.2)

# Free memory
tf.keras.backend.clear_session()


========== Training MobileNetV2 ==========
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Phase 1 — train head
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 308s 4s/step - accuracy: 0.3028 - loss: 1.7881 - val_accuracy: 0.7720 - val_loss: 0.7682 - learning_rate: 1.0000e-04
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.6761 - loss: 0.8556 - val_accuracy: 0.8080 - val_loss: 0.5908 - learning_rate: 1.0000e-04
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.7623 - loss: 0.6552 - val_accuracy: 0.8400 - val_loss: 0.5129 - learning_rate: 1.0000e-04
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.8035 - loss: 0.5318 - val_accuracy: 0.8320 - val_loss: 0.5018 - learning_rate: 1.0000e-04
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.8293 - loss: 0.4740 - val_accuracy: 0.8320 - val_loss: 0.4547 - learning_rate: 1.0000e-04
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 131s 2s/step - accuracy: 0.8402 - loss: 0.4255 - val_accuracy: 0.8360 - v

Saved model to /content/drive/MyDrive/Wheat_Output/mobilenetv2_balanced_500.h5


TypeError: this __dict__ descriptor does not support '_DictWrapper' objects

In [ ]:
model.save("mobilenetv2_balanced_500.keras")

NameError: name 'model' is not defined

In [ ]:
# # 8. Inference helper (consistent with training preprocessing)

# def preprocess_input_image(image_path):
#     img = Image.open(image_path)
#     img = ImageOps.exif_transpose(img)
#     img = img.convert('RGB')
#     w,h = img.size
#     mn = min(w,h)
#     left = (w-mn)//2
#     top = (h-mn)//2
#     img = img.crop((left, top, left+mn, top+mn))
#     img = img.resize((224,224), Image.Resampling.LANCZOS)
#     arr = np.array(img).astype(np.float32)
#     return arr

# # Example load & predict (note: model contains Lambda layer with preprocess; when loading later use custom_objects)
# from tensorflow.keras.models import load_model

# # Determine the correct preprocess_input function based on the model_name
# preprocess_fn_for_loading = None
# model_name_lower = model_name.lower()
# if 'efficientnetv2b0' in model_name_lower:
#     preprocess_fn_for_loading = tf.keras.applications.efficientnet_v2.preprocess_input
# elif 'efficientnetb0' in model_name_lower:
#     preprocess_fn_for_loading = tf.keras.applications.efficientnet.preprocess_input
# elif 'mobilenetv2' in model_name_lower:
#     preprocess_fn_for_loading = tf.keras.applications.mobilenet_v2.preprocess_input
# elif 'resnet50' in model_name_lower:
#     preprocess_fn_for_loading = tf.keras.applications.resnet.preprocess_input

# # Create custom_objects dictionary, using the name 'preprocess' as defined in the Lambda layer
# custom_objects_for_loading = {'preprocess': preprocess_fn_for_loading} if preprocess_fn_for_loading else None

# mp = os.path.join(MODEL_OUTPUT_DIR, f"{model_name_lower}_balanced_{TARGET_PER_CLASS}.h5")
# print('Loading model from', mp)
# model_loaded = load_model(mp, custom_objects=custom_objects_for_loading, compile=False)

# sample_path = test_paths[0]
# img_arr = preprocess_input_image(sample_path)
# pred = model_loaded.predict(np.expand_dims(img_arr, 0))
# print('Pred shape:', pred.shape, ' -> class:', np.argmax(pred, axis=1)[0])

In [ ]:
# =========================
# FINAL INFERENCE CELL
# =========================

import os
import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow.keras.models import load_model

# --------------------------------
# Image preprocessing (SAME as training)
# --------------------------------
def preprocess_input_image(image_path):
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")

    w, h = img.size
    min_dim = min(w, h)
    left = (w - min_dim) // 2
    top = (h - min_dim) // 2
    img = img.crop((left, top, left + min_dim, top + min_dim))

    img = img.resize((224, 224), Image.Resampling.LANCZOS)
    img = np.array(img).astype(np.float32)

    return img  # preprocessing happens INSIDE the model


# --------------------------------
# Load trained model (.keras)
# --------------------------------
# Constants
BASE_PATH = '/content/drive/MyDrive/Wheat_Images'    # original dataset root (per-class folders)
SAVE_ROOT = '/content/drive/MyDrive/Wheat_Final'    # where balanced train/val/test folders will be created
OUTPUT_BASE = '/content/drive/MyDrive'               # where model outputs and splits are saved
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_BASE, 'Wheat_Output')
SPLITS_DIR = os.path.join(OUTPUT_BASE, 'wheat_splits')
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_BASE, 'Wheat_Output')

MODEL_NAME = 'MobileNetV2'

# Determine the correct preprocess_input function based on the model_name
preprocess_fn_for_loading = None
model_name_lower = MODEL_NAME.lower()
if 'efficientnetv2b0' in model_name_lower:
    preprocess_fn_for_loading = tf.keras.applications.efficientnet_v2.preprocess_input
elif 'efficientnetb0' in model_name_lower:
    preprocess_fn_for_loading = tf.keras.applications.efficientnet.preprocess_input
elif 'mobilenetv2' in model_name_lower:
    preprocess_fn_for_loading = tf.keras.applications.mobilenet_v2.preprocess_input
elif 'resnet50' in model_name_lower:
    preprocess_fn_for_loading = tf.keras.applications.resnet.preprocess_input

# Create custom_objects dictionary, using the name 'preprocess_input' as found in the error message
custom_objects_for_loading = {}
if preprocess_fn_for_loading:
    custom_objects_for_loading['preprocess_input'] = preprocess_fn_for_loading

# Construct the full path to the .keras model file
model_path_to_load = os.path.join(MODEL_OUTPUT_DIR, f"{model_name_lower}_balanced_500.keras")
print('Loading model from', model_path_to_load)

# Load the model with custom objects
model_loaded = load_model(model_path_to_load, custom_objects=custom_objects_for_loading)
print('Model loaded successfully.')


Loading model from /content/drive/MyDrive/Wheat_Output/mobilenetv2_balanced_500.keras


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {}}.

Exception encountered: <class 'keras.src.layers.core.lambda_layer.Lambda'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.layers', 'class_name': 'Lambda', 'config': {'name': 'preprocess', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'function': {'modul

In [ ]:
# 9. Evaluation function (accuracy, classification report, confusion matrix)

def evaluate_model(model, test_dataset, class_names):
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
    import seaborn as sns

    y_true = []
    y_pred = []

    for images, labels in test_dataset:
        preds = model.predict(images, verbose=0)
        preds_class = np.argmax(preds, axis=1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds_class)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    print('\nTest Accuracy:', round(acc * 100, 2), '%')
    print('\nClassification report:')
    from sklearn.metrics import classification_report
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion matrix')
    plt.show()

# Run evaluation
evaluate_model(model_loaded, test_ds, CLASS_NAMES)

In [ ]:
# 10. Notes & Next steps
# - If you want to run another model, change `model_name` above and re-run the training cell.
# - Use the saved numpy splits in SPLITS_DIR to recreate datasets exactly the same later.
# - When loading models that include the Lambda(preprocess) layer, you may need to pass
#   custom_objects if the preprocessing function isn't auto-serializable. We saved SavedModel
#   (tf.saved_model.save) which should be easier to load for inference later.
# - If you want inference preprocessing embedded outside the model (so the saved model is pure Keras
#   layers without Lambda), move preprocess_fn into the dataset pipeline instead of Lambda.